# Agent training

In [ ]:
from circle_environment import CircleEnv
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from datetime import datetime
import os
import logging
        
def configure_logging(log_file_path, console_log_level=logging.INFO):
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    handlers = []
    if console_log_level is not None:
        # define a Handler which writes INFO messages or higher to the sys.stderr
        console_handler = logging.StreamHandler()
        console_handler.setLevel(console_log_level)
        console_handler.setFormatter(formatter)
        handlers.append(console_handler)
    if log_file_path is not None:
        # create file handler which logs even debug messages
        file_handler = logging.FileHandler(log_file_path, mode="a")
        file_handler.setLevel(logging.DEBUG)
        file_handler.setFormatter(formatter)
        handlers.append(file_handler)
    # add the handlers to the (root) logger
    logging.basicConfig(level=logging.DEBUG, 
                    handlers=handlers,
                    force=True) # force=True overwrites the logging configuration so that we can change the logfile name

def get_log_level(training_or_evaluation):
    match training_or_evaluation:
        case "training":
            return None
        case "evaluation":
            return logging.INFO
        case _:
            return None

def setup_logging(algorithm, version, map, n_vehicles, training_or_evaluation, model_load_path=None, execution_context="local"):
    # Create a unique identifier for this training run
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    log_id = f"{version}_{map}_{algorithm}_{n_vehicles}vehicles_{training_or_evaluation}_{current_time}"    
    new_model_id = f"{version}_{map}_{algorithm}_{current_time}"
    # if a model path is given, i.e. an existing model is evaluated or trained further
    if model_load_path is not None:
        current_model_dir = model_load_path.split("/")[-2] #.rsplit(".", 1)[0] # Extract directory from model_load_path
    else:
        current_model_dir = new_model_id
    
    # Set up non existing directories
    match execution_context:
        case "local":
            root = "."
        case "colab":
            root = "/content/drive/MyDrive/Colab Notebooks/rl-charging-allocation"
        case _:
            raise ValueError(f"Invalid execution context {execution_context}. Must be 'local' or 'colab'.")

    model_dir = f"{root}/models/{current_model_dir}"
    log_dir = f"{model_dir}/{training_or_evaluation}"
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    
    model_save_path = f"{model_dir}/{new_model_id}.zip"
    py_log_path = f"{log_dir}/{log_id}.log"
    console_log_level = get_log_level(training_or_evaluation)
    configure_logging(log_file_path=py_log_path, console_log_level=console_log_level)
    return model_dir, model_save_path

def train_model(algorithm, policy, version, map, n_vehicles, n_steps, execution_context="local"):
    # initiate environment
    env = CircleEnv(render_mode=None, vehicles_to_spawn=n_vehicles)
    # setup logging
    model_dir, model_save_path = setup_logging(algorithm, version, map, n_vehicles, "training", execution_context=execution_context)
    # Train the agent
    match algorithm:
        case "PPO":
            model = PPO(policy, env, verbose=1, tensorboard_log=model_dir)
        case "A2C":
            model = A2C(policy, env, verbose=1, tensorboard_log=model_dir)
        case "DQN":
            model = DQN(policy, env, verbose=1, tensorboard_log=model_dir)
        case _:
            raise ValueError("Invalid model type")
    tb_log_name = "tensorboard"
    try:
        model.learn(n_steps, tb_log_name=tb_log_name, callback=CustomTensorboardCallback())
        model.save(model_save_path)
        env.close()
        return model_save_path
    except Exception as e:
        model.save(model_save_path)
        env.close()
        raise e
    
def load_model(model_path, algorithm, env):
    match algorithm:
        case "PPO":
            model = PPO.load(model_path, env=env)
        case "A2C":
            model = A2C.load(model_path, env=env)
        case "DQN":
            model = DQN.load(model_path, env=env)
        case _:
            raise ValueError("Invalid model type")
    return model

def further_train_model(model_load_path, algorithm, version, map, n_vehicles, n_steps, execution_context="local"):
    # initiate environment
    env = CircleEnv(render_mode=None, vehicles_to_spawn=n_vehicles)
    # setup logging
    model_dir, model_save_path = setup_logging(algorithm, version, map, n_vehicles, "training", model_load_path, execution_context=execution_context)
    # Train the agent
    model = load_model(model_load_path, algorithm, env)
    tb_log_name = "tensorboard"
    try:
        model.learn(n_steps, tb_log_name=tb_log_name, callback=CustomTensorboardCallback())
        model.save(model_save_path)
        env.close()
        return model_save_path
    except Exception as e:
        model.save(model_save_path)
        env.close()
        raise e

def evaluate_model(model_load_path, algorithm, version, map, n_vehicles, n_episodes, execution_context="local"):
    # initiate environment
    env = CircleEnv(render_mode="human", vehicles_to_spawn=n_vehicles)
    # setup logging
    setup_logging(algorithm, version, map, n_vehicles, "evaluation", model_load_path, execution_context=execution_context)
    # Load saved model
    model = load_model(model_load_path, algorithm, env)
    # Evaluate the agent
    try:
        mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=n_episodes)
        print(f"mean_reward: {mean_reward}, std_reward: {std_reward}")
        env.close()
    except KeyboardInterrupt:
        env.close()

class CustomTensorboardCallback(BaseCallback):
    """ used to log the current agent step number """
    def __init__(self, verbose=0):
        super(CustomTensorboardCallback, self).__init__(verbose)
    
    def _on_step(self) -> bool:
        # TODO: num_timesteps an env weitergeben, damit der env-logger auch die sb3-timesteps (self.num_timesteps) loggen kann. Dann braucht der sb3-Logger das auch nicht mehr mitloggen.
        # self.logger.record("current_step", self.num_timesteps)
        logger = logging.getLogger("rl")
        logger.debug(f"Agent Step {self.num_timesteps}")

        # How many times is a car charging during an episode (on average)?
        # Retrieve the variable from the training environment
        env = self.training_env.envs[0]  # For vectorized environments, access the first one
        if hasattr(env, "charging_stops_per_episode_mean"):
            charging_stops_per_episode_mean = env.charging_stops_per_episode_mean
            self.logger.record("env/charging_stops_per_episode_mean", charging_stops_per_episode_mean)

        return True


In [ ]:
# Train new model
model_path = train_model("A2C", "MultiInputPolicy", "v0.5.0", "circle", n_vehicles=5, n_steps=3000, execution_context="local")

In [ ]:
# manually configure model load path
model_dir = "v0.5.0_circle_DQN_2024-12-10_13-42-17"
model_id = "v0.5.0_circle_DQN_2024-12-10_13-42-17"
model_path = f"models/{model_dir}/{model_id}.zip"

In [ ]:
# Train saved model further
model_path = further_train_model(model_path, "DQN", "v0.5.0", "circle", n_vehicles=5, n_steps=20000)

In [ ]:
# Quick evaluation
# model_path = "models/v0.3_circle_a2c_2024-09-18_23-58-11/v0.3_circle_a2c_2024-09-18_23-58-11.zip"
evaluate_model(model_path, "DQN", "v0.5.0", "circle", n_vehicles=5, n_episodes=5)

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from circle_environment import CircleEnv

# Observe execution of trained agent in GUI
env = CircleEnv(render_mode="human", vehicles_to_spawn=5)
model = A2C.load(model_save_name, env)

num_steps = 5
observation, info = env.reset()
for t in range(num_steps):
        actions, _ = model.predict(observation, state=None, deterministic=False)
        observation, reward, terminated, truncated, info = env.step(actions)

env.close()

In [ ]:
# Random actions to compare with the agent
env = CircleEnv(render_mode="human", vehicles_to_spawn=5)
observation, info = env.reset()
try:
    for _ in range(5):
        action = env.action_space.sample() # select a random action
        observation, reward, terminated, truncated, info = env.step(action)
        # if terminated or truncated:
            # observation, info = env.reset()
finally:        
    env.close()

In [ ]:
# --- Benchmark environment ---

def benchmark_env(random_seed):
    env = CircleEnv(render_mode=None, vehicles_to_spawn=15)
    # set seed for reproducability
    import random
    random.seed(random_seed) # needed for batteries of simulation class TODO: make this seedable via the env.seed of gymnasium
    observation, info = env.reset(seed=random_seed)
    env.action_space.seed(random_seed)
    try:
        for _ in range(200):
            action = env.action_space.sample() # select a random action
            observation, reward, terminated, truncated, info = env.step(action)
            if terminated or truncated:
                observation, info = env.reset()
    finally:
        env.close()
    
import cProfile
# cProfile.run('train_model("A2C", "MultiInputPolicy", "v0.3", "circle", n_vehicles=5, n_steps=10000)', 'output.pstats')
benchmark_file = 'output_v0.3.1.pstats'
cProfile.run('benchmark_env(random_seed=1)', benchmark_file)

In [ ]:
import pstats
from pstats import SortKey
p = pstats.Stats(benchmark_file)
p.sort_stats(SortKey.CUMULATIVE).print_stats()